In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("combined_concrete_data.csv")

df.head()

,binder,extra water,alkaline solution,molarity of mix,fine aggregate,coarse aggregate,age,curing temperature,compressive strength,water,pozzolan,foaming agent,density,concrete type,water binder ratio
0,428.0,43.0,171.0,14.0,630.0,1170.0,1.0,75.0,30.0,NaN,NaN,NaN,NaN,0,0.500000
1,444.0,43.0,155.0,14.0,630.0,1170.0,1.0,75.0,30.0,NaN,NaN,NaN,NaN,0,0.445946
2,428.0,43.0,171.0,14.0,630.0,1170.0,1.0,75.0,40.0,NaN,NaN,NaN,NaN,0,0.500000
3,428.0,43.0,171.0,14.0,630.0,1170.0,1.0,75.0,28.0,NaN,NaN,NaN,NaN,0,0.500000
4,428.0,43.0,171.0,14.0,630.0,1170.0,2.0,75.0,32.0,NaN,NaN,NaN,NaN,0,0.500000


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
binder,2600.0,444.672208,136.268401,107.200000,400.000000,405.000000,468.000000,992.800000
extra water,1691.0,17.955802,26.448564,0.000000,0.000000,0.000000,32.000000,128.570000
alkaline solution,1691.0,187.283390,36.738527,106.680000,160.020000,180.000000,200.000000,294.400000
molarity of mix,1691.0,11.519456,2.776044,4.100000,10.000000,12.000000,14.000000,20.000000
fine aggregate,2600.0,631.342854,178.804817,0.000000,550.000000,645.000000,697.400000,1098.000000
coarse aggregate,1691.0,1099.968297,188.269756,647.800000,989.500000,1181.000000,1220.350000,1369.000000
age,2600.0,25.490000,22.438885,1.000000,7.000000,28.000000,28.000000,91.000000
curing temperature,1691.0,34.972797,16.904233,20.000000,27.000000,27.000000,28.000000,80.000000
compressive strength,2600.0,29.266384,21.230027,0.080000,10.422500,26.735575,43.745000,110.000000
water,909.0,232.404999,83.601458,68.900000,169.000000,234.000000,290.325000,484.000000


I am going to train with the models, which innately support NANs in the dataset(lightgbm, xgboost), and then impute 0 valuse to the NANs and train other models.

In [4]:
target = "compressive strength"

X = df.drop(columns=[target])
y = df[target]

In [5]:
from sklearn.model_selection import train_test_split

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=df['concrete type'] 
)

In [6]:
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (2080, 14), Test: (520, 14)


In [7]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import pandas as pd

# ==========================================
# PATH 1: For XGBoost / LightGBM / CatBoost 
# Objective: Scale the data, but PRESERVE NaNs
# ==========================================
scaler_nan = StandardScaler()

X_train = pd.DataFrame(scaler_nan.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler_nan.transform(X_test), columns=X_test.columns)


# ==========================================
# PATH 2: For Random Forest / Gradient Boosting 
# Objective: TRUE Zero Imputation, THEN scale
# ==========================================
imputer_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

X_train_imputed = pd.DataFrame(imputer_pipeline.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer_pipeline.transform(X_test), columns=X_test.columns)

print(f"Path 1 (NaNs Preserved) -> Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Path 2 (Zero Imputed)   -> Train: {X_train_imputed.shape}, Test: {X_test_imputed.shape}")


Path 1 (NaNs Preserved) -> Train: (2080, 14), Test: (520, 14)
Path 2 (Zero Imputed)   -> Train: (2080, 14), Test: (520, 14)


In [12]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor

In [15]:
# XGB REGRESSOR

xgb_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    missing=np.nan,           
    enable_categorical=True   
)

param_grid_xgb = {
    'n_estimators': [300, 500],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
grid_search_xgb.fit(X_train, y_train)

best_xgb_model = grid_search_xgb.best_estimator_
print("Best params:", grid_search_xgb.best_params_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


Best params: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 300, 'subsample': 1.0}


In [16]:
# LGBM REGRESSOR

import optuna
from sklearn.model_selection import cross_val_score, KFold
import logging

def lgbm_objective(trial):
    params = {
        # Search space
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'boosting_type': 'gbdt',
        'verbose': -1,
        'random_state': 42,
        'n_jobs': -1 # Let LightGBM use all cores
    }
    
    model = LGBMRegressor(**params)
    
    # Example using 5-fold CV for speed during tuning
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_squared_error', n_jobs=1)
    
    return scores.mean()

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lgbm_objective, n_trials=150, show_progress_bar=False) 

print("Best params:", study_lgbm.best_params)


[I 2026-03-30 16:02:56,307] A new study created in memory with name: no-name-d6f77a9b-c8cf-44b8-9e6e-a29fe7d74687
[I 2026-03-30 16:03:02,917] Trial 0 finished with value: -43.44932206482934 and parameters: {'n_estimators': 259, 'learning_rate': 0.17795946196765802, 'num_leaves': 55}. Best is trial 0 with value: -43.44932206482934.
[I 2026-03-30 16:03:04,958] Trial 1 finished with value: -43.02645573383893 and parameters: {'n_estimators': 247, 'learning_rate': 0.10078931745539338, 'num_leaves': 23}. Best is trial 1 with value: -43.02645573383893.
[I 2026-03-30 16:03:09,139] Trial 2 finished with value: -44.93990716292189 and parameters: {'n_estimators': 227, 'learning_rate': 0.04497863490526672, 'num_leaves': 56}. Best is trial 1 with value: -43.02645573383893.
[I 2026-03-30 16:03:14,117] Trial 3 finished with value: -47.61386156668939 and parameters: {'n_estimators': 297, 'learning_rate': 0.024214682716518707, 'num_leaves': 65}. Best is trial 1 with value: -43.02645573383893.
[I 2026-0

Best params: {'n_estimators': 207, 'learning_rate': 0.1737559988817346, 'num_leaves': 27}


I implemented the optuna hyperparameter tuning for lightgbm, and it is giving me a good boost in the performance. I will implement it for xgboost as well, and then move on to the imputation part. I implemented the optuna hyperparameter tuning, because the gridsearchcv was taking a lot of time to run, like 5 candidates initialization in gridsearchcv was taking around 5.37 ish minutes, and I had around 125+ candidates to try. So optune is a good choice for hyperparameter tuning in this case, because it is much faster than gridsearchcv, and it also gives better results. I will not implement it to others (for now), if situation arises, I will turn to optuna.

In [ ]:
# CATBOOST REGRESSOR

from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(random_state=42, verbose=0)

param_grid_catboost = {'iterations': [500, 800], 'depth': [6,8,10],
              'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]}

grid_catboost = GridSearchCV(catboost_model, param_grid_catboost, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_catboost.fit(X_train, y_train)
best_catboost_model = grid_catboost.best_estimator_

Fitting 5 folds for each of 24 candidates, totalling 120 fits


In [ ]:
# GRADIENT BOOSTING REGRESSOR

from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(random_state=42)

param_grid_gbr = {'n_estimators': [300, 500], 'max_depth': [5,7],
              'learning_rate': [0.05, 0.1]}

grid_gradient_boost = GridSearchCV(model, param_grid_gbr, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_gradient_boost.fit(X_train_imputed, y_train)
best_model_gradient_boost = grid_gradient_boost.best_estimator_


In [ ]:
# RANDOMFOREST REGRESSOR

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid_rf = {'n_estimators': [200, 300], 'max_depth': [10, 15, 20],
              'min_samples_split': [2, 5]}

grid_random_forest = GridSearchCV(model, param_grid_rf, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_random_forest.fit(X_train_imputed, y_train)
best_model_rf = grid_random_forest.best_estimator_


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2 Score": r2_score(y_true, y_pred)
    }

In [ ]:
y_pred_xgb = best_xgb_model.predict(X_test)

xgb_eval = evaluate_model(y_test, y_pred_xgb)

print(xgb_eval)

In [ ]:
y_pred_catboost = best_catboost_model.predict(X_test)

catboost_eval = evaluate_model(y_test, y_pred_catboost)

print(catboost_eval)

In [ ]:
best_lgbm_params = study.best_params

lgbm_model = LGBMRegressor(
    boosting_type="gbdt",
    random_state=42,
    verbose=-1,
    **best_lgbm_params
)

lgbm_model.fit(X_train, y_train)

y_pred_lgbm = lgbm_model.predict(X_test)

lgbm_eval = evaluate_model(y_test, y_pred_lgbm)

print(lgbm_eval)

In [ ]:
y_pred_gradient_boost = best_model_gradient_boost.predict(X_test_imputed)

gradient_boost_eval = evaluate_model(y_test ,y_pred_gradient_boost)

print(gradient_boost_eval)

In [ ]:
y_pred_rf = best_model_rf.predict(X_test_imputed)

rf_eval = evaluate_model(y_test, y_pred_rf)

print(rf_eval)